# Clase 206 — Data testing con Great Expectations + Pandera

Definimos un suite de expectations sobre California Housing y validamos con (a) Great Expectations 1.x, (b) Pandera, (c) Polars + checks ad-hoc. Inyectamos un bug a propósito para ver el sistema detectarlo.

In [ ]:
import pandas as pd, numpy as np
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.data.copy()
df['target'] = data.target
print(df.shape, '\n', df.describe().round(2).T[['min', 'max', 'mean']])

## 1. Great Expectations 1.x (in-process)

In [ ]:
# Requiere: pip install great-expectations>=1.0
try:
    import great_expectations as gx
    ctx = gx.get_context(mode='ephemeral')
    src = ctx.data_sources.add_pandas('demo')
    asset = src.add_dataframe_asset(name='housing')
    batch_def = asset.add_batch_definition_whole_dataframe('all')
    batch = batch_def.get_batch(batch_parameters={'dataframe': df})

    from great_expectations.expectations import (
        ExpectColumnValuesToNotBeNull, ExpectColumnValuesToBeBetween,
        ExpectTableRowCountToBeBetween, ExpectColumnPairValuesAToBeGreaterThanB,
    )
    suite = ctx.suites.add(gx.ExpectationSuite(name='housing_v1'))
    suite.add_expectation(ExpectTableRowCountToBeBetween(min_value=1000, max_value=100000))
    suite.add_expectation(ExpectColumnValuesToNotBeNull(column='MedInc'))
    suite.add_expectation(ExpectColumnValuesToBeBetween(column='MedInc', min_value=0, max_value=20))
    suite.add_expectation(ExpectColumnValuesToBeBetween(column='HouseAge', min_value=0, max_value=200))
    suite.add_expectation(ExpectColumnPairValuesAToBeGreaterThanB(column_A='AveRooms', column_B='AveBedrms'))

    result = batch.validate(suite)
    print(f'success: {result.success}, statistics: {result.statistics}')
    for r in result.results:
        flag = '✅' if r.success else '❌'
        print(f'{flag} {r.expectation_config.type}')
except ImportError:
    print('pip install great-expectations para esta celda')

## 2. Pandera — sintaxis más concisa

In [ ]:
import pandera as pa
from pandera import Column, Check, DataFrameSchema

schema = DataFrameSchema({
    'MedInc': Column(float, [Check.ge(0), Check.le(20), Check.not_null()]),
    'HouseAge': Column(float, [Check.ge(0), Check.le(200)]),
    'AveRooms': Column(float, Check.ge(0)),
    'AveBedrms': Column(float, Check.ge(0)),
    'target': Column(float, Check.ge(0)),
}, checks=[
    Check(lambda d: (d['AveRooms'] >= d['AveBedrms']).all(), name='rooms_ge_bedrooms'),
    Check(lambda d: len(d) > 1000, name='min_rows'),
])

validated = schema.validate(df, lazy=True)
print('OK — todas las checks pasaron')
print('shape:', validated.shape)

## 3. Inyectar un bug y ver el sistema detectarlo

In [ ]:
df_corrupt = df.copy()
df_corrupt.loc[df_corrupt.sample(50, random_state=1).index, 'MedInc'] = np.nan
df_corrupt.loc[df_corrupt.sample(10, random_state=2).index, 'HouseAge'] = -5
df_corrupt.loc[df_corrupt.sample(20, random_state=3).index, 'AveBedrms'] = 999   # más que AveRooms

try:
    schema.validate(df_corrupt, lazy=True)
    print('UNEXPECTED: validation passed')
except pa.errors.SchemaErrors as e:
    print(f'❌ {len(e.failure_cases)} check failures (esperado):')
    print(e.failure_cases.head(10).to_string(index=False))

## 4. Integración con pipeline — gate por exit code

In [ ]:
validate_script = '''\
# validate.py — corre como step del DVC/Airflow/Prefect/CI
import sys, json, pandas as pd, pandera as pa
from schemas import housing_schema   # mismo schema en módulo compartido

df = pd.read_parquet(sys.argv[1])
try:
    housing_schema.validate(df, lazy=True)
    print(json.dumps({"status": "ok", "rows": len(df)}))
    sys.exit(0)
except pa.errors.SchemaErrors as e:
    print(json.dumps({"status": "failed", "errors": e.failure_cases.head(20).to_dict(orient="records")}))
    sys.exit(1)   # exit != 0 → pipeline aborta
'''
print(validate_script)

## Ejercicio guiado

1. Agregá una expectation: `target` tiene que tener `mean` entre `last_week_mean ± 10%` (gate de drift relativo).
2. Configurá Great Expectations con un Data Docs HTML. Hospedalo en GitHub Pages desde un GH Actions workflow.
3. Convertí el `validate.py` en un step del DVC pipeline (Clase 194) que corre antes del training. Verificá que si la validación falla, `dvc repro` aborta.
4. Para datasets grandes, instalá PyDeequ + Spark y replicá el suite. Compará performance.
5. Suite separado en críticas (abortan pipeline) vs warnings (loggean pero no abortan).

## Conclusiones

- Data testing es ortogonal a unit testing: el código puede estar OK pero la data llegar rota.
- Pandera es más DX-friendly para Python puro; GE es más completo para auditoría.
- El suite es contrato: va a git, va por PR.
- Validation gate antes de training evita que un mes de retrainings de data sucia llegue a producción.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. Great Expectations y Pandera no están instalados, así que mostramos su **API real** y ejecutamos el *concepto* con pandas puro: perfilar un dataset, implementar cada *expectation* como una aserción, correr un "checkpoint" que colecta fallos, renderizar un mini Data Docs HTML y comparar con el equivalente Pandera. Testear datos = las mismas reglas que un `assert`, pero declarativas y con reporte.

In [ ]:
import pandas as pd, numpy as np
from sklearn.datasets import fetch_california_housing
df = fetch_california_housing(as_frame=True).frame
print('dataset:', df.shape)
print(df[['MedInc', 'AveRooms', 'AveBedrms']].describe().round(2))

### Ejercicio 1 — Bootstrap de un suite (profiling)

`gx init` + profiling genera *expectations* candidatas a partir de estadísticas del dataset (min/max/nulos/tipos). Lo replicamos: perfilamos y proponemos expectations razonables.

In [ ]:
# API REAL: gx init ; gx datasource new ; profiler -> suite candidata
def profile_to_expectations(df):
    exp = []
    for col in df.select_dtypes('number').columns:
        lo, hi = df[col].min(), df[col].max()
        exp.append({'expectation': 'expect_column_values_to_be_between',
                    'column': col, 'min': round(float(lo), 2), 'max': round(float(hi), 2)})
        if df[col].isna().sum() == 0:
            exp.append({'expectation': 'expect_column_values_to_not_be_null', 'column': col})
    return exp

suite = profile_to_expectations(df)
print(f'{len(suite)} expectations candidatas. Ejemplo:')
for e in suite[:3]:
    print('  ', e)
assert any(e['column'] == 'MedInc' for e in suite)
print('OK — profiling propone; el ingeniero revisa (borra las absurdas, ajusta rangos).')

### Ejercicio 2 — Custom expectations (implementadas en pandas)

Tres reglas del README, cada una como aserción verificable: rango de `MedInc`, conteo de filas, y la relación de par `AveRooms > AveBedrms`.

In [ ]:
# API REAL de GE:
#   validator.expect_column_values_to_be_between("MedInc", 0, 20)
#   validator.expect_table_row_count_to_be_between(1000, 100000)
#   validator.expect_column_pair_values_A_to_be_greater_than_B("AveRooms", "AveBedrms")
def expect_between(df, col, lo, hi):
    frac = df[col].between(lo, hi).mean()
    return {'expectation': f'{col} in [{lo},{hi}]', 'success': bool(frac == 1.0), 'pct_ok': round(frac, 4)}
def expect_row_count(df, lo, hi):
    return {'expectation': f'row_count in [{lo},{hi}]', 'success': bool(lo <= len(df) <= hi), 'n': len(df)}
def expect_A_gt_B(df, a, b):
    frac = (df[a] > df[b]).mean()
    return {'expectation': f'{a} > {b}', 'success': bool(frac == 1.0), 'pct_ok': round(frac, 4)}

results = [expect_between(df, 'MedInc', 0, 20),
           expect_row_count(df, 1000, 100000),
           expect_A_gt_B(df, 'AveRooms', 'AveBedrms')]
for r in results:
    print(r)
assert results[1]['success'] is True
print('OK — expectations = aserciones declarativas con % de cumplimiento.')

### Ejercicio 3 — Checkpoint (corre el suite y alerta si falla)

Un checkpoint corre todas las expectations y, si alguna falla, dispara una alerta. Lo implementamos con un suite que incluye una regla que **falla a propósito** para ver el alertado.

In [ ]:
def run_checkpoint(df, suite_fns):
    results = [fn(df) for fn in suite_fns]
    failed = [r for r in results if not r['success']]
    alert = None
    if failed:
        alert = f'❌ Checkpoint FAILED: {len(failed)}/{len(results)} expectations. ' \
                f'Primera: {failed[0]["expectation"]}'   # aquí iría el POST a Slack
    return {'success': len(failed) == 0, 'failed': failed, 'alert': alert}

suite_fns = [lambda d: expect_between(d, 'MedInc', 0, 20),
             lambda d: expect_between(d, 'MedInc', 0, 5)]   # <- fallará (hay valores > 5)
out = run_checkpoint(df, suite_fns)
print('checkpoint success:', out['success'])
print('alert:', out['alert'])
assert out['success'] is False and out['alert'] is not None
print('OK — un checkpoint verde/rojo es lo que corrés en CI antes de entrenar.')

### Ejercicio 4 — Data Docs (mini reporte HTML)

`gx docs build` genera un HTML navegable con el suite + el último resultado. Renderizamos una versión mínima del reporte para "mostrarle a un PM/regulador".

In [ ]:
def render_data_docs(results):
    rows = ''.join(
        f'<tr><td>{r["expectation"]}</td>'
        f'<td style="color:{"green" if r["success"] else "red"}">{"PASS" if r["success"] else "FAIL"}</td></tr>'
        for r in results)
    return f'<html><body><h2>Validation Result</h2><table border="1">{rows}</table></body></html>'

html = render_data_docs([expect_between(df, 'MedInc', 0, 20),
                         expect_A_gt_B(df, 'AveRooms', 'AveBedrms')])
print(html[:200], '...')
assert '<table' in html and 'PASS' in html
print('\nOK — Data Docs vuelve auditable la calidad de datos (evidencia para compliance).')

### Ejercicio 5 — Alternativa con Pandera

Pandera define el esquema como una clase; validar es una línea. Mostramos el `DataFrameModel` real y ejecutamos el equivalente en pandas para comparar verbosidad/velocidad.

In [ ]:
# API REAL de Pandera:
#   import pandera as pa
#   from pandera.typing import Series
#   class HousingSchema(pa.DataFrameModel):
#       MedInc: Series[float] = pa.Field(in_range={"min_value": 0, "max_value": 20})
#       AveRooms: Series[float] = pa.Field(gt=0)
#   HousingSchema.validate(df)
schema = {'MedInc': {'in_range': (0, 20)}, 'AveRooms': {'gt': 0}, 'HouseAge': {'in_range': (0, 60)}}
def validate_schema(df, schema):
    errors = []
    for col, rules in schema.items():
        if 'in_range' in rules:
            lo, hi = rules['in_range']
            if not df[col].between(lo, hi).all():
                errors.append(f'{col} fuera de [{lo},{hi}]')
        if 'gt' in rules and not (df[col] > rules['gt']).all():
            errors.append(f'{col} no siempre > {rules["gt"]}')
    return errors

errors = validate_schema(df, schema)
print('errores de esquema:', errors if errors else 'ninguno')
assert errors == [], 'el dataset cumple el esquema definido'
print('OK — Pandera: menos verboso que GE, ideal para validación inline en el pipeline.')